![image.png](https://i.imgur.com/a3uAqnb.png)


# **Day 2 Lab 3: Multi-Class Classification**

---
In this lab, you'll learn how to train ML Models for **multi-class classification**

# 📊 **Data**
https://www.kaggle.com/datasets/mujtabamatin/air-quality-and-pollution-assessment

> This dataset contains factors that influence pollution, and asks you to classify the **Air Quality Level**.
>
> The target classes are described as: **Good, Moderate, Poor, Hazardous**.
>
> This is a **Multiclass-Classification** problem.

The data contains the following columns:

* Temperature: Temperature in °C.
* Humidity: Relative humidity (%).
* PM2.5: Fine particulate matter concentration (µg/m³).
* PM10: Coarse particulate matter concentration (µg/m³).
* NO2: Nitrogen dioxide concentration.
* SO2: Sulfur dioxide concentration.
* CO: Carbon monoxide concentration.
* Proximity to Industrial Areas: Distance to nearest industrial zone (km).
* Population Density: People per unit area (often correlates with emissions/traffic).

> **Target**: Air Quality (e.g., Good / Moderate / Poor / Hazardous).

---

# Task 1: EDA & Data Preprocessing

## 1.1 Import Libraries

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

## 1.2 Read the Data

In [ ]:
# Download latest version
path = kagglehub.dataset_download("mujtabamatin/air-quality-and-pollution-assessment")

print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "updated_pollution_dataset.csv")

df = pd.read_csv(csv_path)

df.head()

In [ ]:
df.info()

## 1.3 Exploratory Data Analysis (EDA)

**Rule of thumb checklist:**

| Question | If YES | If NO |
|----------|--------|-------|
|  **1. Is the target imbalanced?** | Use F1/Precision/Recall + StratifiedKFold | Accuracy is fine + KFold |
|  **2. Missing values?** | Impute or drop | Proceed |
|  **3. Categorical columns?** | Encode them | Proceed |
|  **4. Duplicates?** | Drop them | Proceed |
|  **5. Different scales?** | Normalize | Proceed |

In [ ]:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Air Quality")

> **Rule of thumb:** If class distributions are not equal, then our data is **imbalanced**. Use StratifiedKFold and focus on F1-score. And if data distribution is balanced, StratifiedKFold will act like regular KFold, so always use StratifiedKFold :)

In [ ]:
# 2. Do we have missing values?
def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

> If we have categorical columns. We need to **encode** them (convert them into numbers) for our models.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# Encode the target column
df['Air Quality'] = le.fit_transform(df['Air Quality'])

df

In [ ]:
# 4. Check for duplicates
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Pick only the numerical columns, NOT the target
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Air Quality")

# FIX: no scaling here. X stays unscaled; scaling happens per-fold in Task 4.

df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Pick only the numerical columns, NOT the target
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Air Quality")

# FIX: no scaling here. X stays unscaled; scaling happens per-fold in Task 4.

df.head()

In [ ]:
# Prepare the data as X and y

X = df.drop("Air Quality", axis=1).astype(float)
y = df["Air Quality"].astype(float)

# Task 2: Multiclass Logistic Regression

- **Softmax Function**

Instead of Sigmoid, we use **Softmax**. It "squashes" a vector of raw scores (logits) into a probability distribution where the sum of all probabilities equals **1**:

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$



- **Categorical Cross-Entropy Loss**

For multiclass problems, we use **Categorical Cross-Entropy**. This measures the difference between the predicted probability distribution and the actual "one-hot" encoded label:

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} y_{i,k} \log(\hat{y}_{i,k})$$

Where:
- $m$ = number of samples
- $K$ = number of classes
- $y_{i,k}$ = binary indicator (0 or 1) if class $k$ is the correct label for sample $i$
- $\hat{y}_{i,k}$ = predicted probability for class $k$

> **Intuition:** The loss only cares about the probability assigned to the **correct** class. If the model assigns a low probability to the true class, the penalty becomes very large!

In [ ]:
# The softmax function
def softmax(z):
  z_shifted = z - np.max(z, axis=1, keepdims=True)
  exp_z = np.exp(z_shifted)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [ ]:
# Categorical cross entropy loss function using NumPy
def categorical_cross_entropy(y, y_hat):
  epsilon = 1e-15
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon)

  loss = -np.mean(np.sum(y * np.log(y_hat), axis=1))
  return loss

- **One-Hot Encoding**

> For multi-class, we need to convert labels [0, 1, 2, 3] into one-hot vectors:
> - Class 0: `[1, 0, 0, 0]`
> - Class 1: `[0, 1, 0, 0]`
> - Class 2: `[0, 0, 1, 0]`
> - Class 3: `[0, 0, 0, 1]`

In [ ]:
def one_hot_encode(y, num_classes):
    y = np.array(y)
    m = len(y)
    # 1. Create a grid of all zeros (num_samples, num_classes)
    one_hot = np.zeros((m, num_classes))

    # 2. Go through each sample one by one
    for i in range(m):
        # Identify which class this sample belongs to
        class_label = int(y[i])

        # In this row (i), set the specific class column to 1
        one_hot[i, class_label] = 1

    return one_hot

In [ ]:
def gradient_descent(X, y, num_classes, lr, n_iters=1000):
  # Get the number of samples (m) and number of features (n)
  m, n = X.shape

  # Initialize weight matrix with shape (n, num_classes)
  theta = np.zeros((n, num_classes))

  # One-hot encode the labels
  y_onehot = one_hot_encode(y, num_classes)

  losses = []

  for _ in tqdm(range(n_iters), desc="Training Multiclass Logistic Regression"):
    # Calculate the logits z
    z = np.dot(X, theta)

    # Get class probabilities using softmax
    y_pred = softmax(z)

    # Compute the gradient of Categorical Cross-Entropy with Softmax
    # ∂J/∂θ = (1/m) * X^T * (y_pred - y_onehot)
    gradient = np.dot(X.T, (y_pred - y_onehot)) / m

    # Update weights
    theta -= lr * gradient

    # Track loss
    loss = categorical_cross_entropy(y_onehot, y_pred)
    losses.append(loss)

  return theta, losses

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
n_splits = 3 # K=3 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
sr_results = {'loss': [], 'acc': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train using gradient descent with learning rate = 0.5
  theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=4)

  # Calculate z & class probabilities for X_test
  z = np.dot(X_test, theta)
  y_pred_proba = softmax(z)

  # Pick the predicted classes with the highest probability
  y_pred = np.argmax(y_pred_proba, axis=1)

  # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  # Store results
  sr_results['loss'].append(losses)
  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)

In [ ]:
#TODO: Calculate the average losses across folds
avg_loss = np.mean(sr_results['loss'], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Training', color='purple')
plt.title('Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Categorical Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print("SOFTMAX REGRESSION Performance:")
# TODO: Print the average of each evaluation metric across folds
print(f"  Accuracy:  {np.mean(sr_results['acc']):.4f}")
print(f"  F1-Score:  {np.mean(sr_results['f1']):.4f}")

---

# Task 3: Define the ML models

In [ ]:
pip install xgboost

In [ ]:
# Import models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
# TODO: Define models with hyperparameters of your choice
models = {
  "LR": LogisticRegression(
      max_iter=1000
  ),
  "KNN": KNeighborsClassifier(
      n_neighbors=3,
  ),
  "SVM": SVC(
      kernel='rbf',
      C=1.0
  ),
  "Decision Tree": DecisionTreeClassifier(
      max_depth=16
  ),
  "Random Forest": RandomForestClassifier(
      n_estimators=200,
      max_depth=10
  ),
  "XGBoost": XGBClassifier(
      verbosity=0,
      n_estimators=280,
      max_depth=10,
  ),
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
}

---

# Task 4: Training with Cross-Validation

In [ ]:
results = {}

for model_name in models:
  results[model_name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  for model_name, model in models.items():

    print(f"Training {model_name}...")

    # FIX: Pipeline fits the scaler on the train fold only — no leakage.
    pipe = Pipeline([
        ("scaler", MinMaxScaler()),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

    results[model_name]['accuracy'].append(accuracy)
    results[model_name]['f1'].append(f1)

**Print the results**

In [ ]:
for model_name in results:
  print(f"\n{model_name}:")
  # Print the average of each evaluation metric across folds
  print(f"  Accuracy:  {np.mean(results[model_name]['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(results[model_name]['f1']):.4f}")

# **Contribution: Sattam Altwaim** :)



![image.jpeg](https://i.imgflip.com/agk7s5.jpg)
